# Day 048 Project: Predictive Model

## What You're Building

A full ML pipeline — from raw housing data to a trained, evaluated LinearRegression model with a saved visualisation.

## Project Requirements

1. Generate the housing dataset with `make_regression_data(200)`
2. Use `FeatureEngineer(target_col='price').fit_transform(df)` to preprocess
3. Train a model with `train_model()` and evaluate with `evaluate_model()`
4. Store results as `r2` and `rmse` variables (floats)
5. Build a coefficient table: which features matter most?
6. Save a scatter plot of actual vs predicted prices to `predictive_model.png`
7. Run `_run_project_checks()` to verify

The deliverable: a saved chart + printed evaluation metrics.

## Provided: All Implementations

In [ ]:
import pandas as pd
import numpy as np
import warnings
from sklearn.model_selection import train_test_split
warnings.filterwarnings('ignore')


def make_regression_data(n: int = 200, seed: int = 42) -> pd.DataFrame:
    """Synthetic housing dataset with one categorical column (neighborhood)."""
    rng = np.random.default_rng(seed)
    area         = rng.uniform(500, 3000, n).round(0)
    bedrooms     = rng.integers(1, 6, n)
    age          = rng.uniform(0, 50, n).round(1)
    neighborhood = rng.choice(['downtown', 'suburb', 'rural'], n)
    price = (
        area * 150
        + bedrooms * 10_000
        - age * 1_000
        + np.where(neighborhood == 'downtown', 50_000, 0)
        + np.where(neighborhood == 'suburb',   20_000, 0)
        + rng.standard_normal(n) * 10_000
    ).round(-2)
    return pd.DataFrame({
        'area':         area.astype(int),
        'bedrooms':     bedrooms,
        'age':          age,
        'neighborhood': neighborhood,
        'price':        price.astype(int),
    })


def prepare_features(df: pd.DataFrame, target_col: str,
                     numeric_only: bool = True):
    """Return (X, y) separating features from target."""
    X = df.drop(columns=[target_col])
    if numeric_only:
        X = X.select_dtypes(include='number')
    y = df[target_col]
    return X, y


def split_data(X: pd.DataFrame, y: pd.Series,
               test_size: float = 0.2,
               random_state: int = 42) -> dict:
    """Wrap train_test_split, return a result dict."""
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state
    )
    return {
        'X_train':    X_train,
        'X_test':     X_test,
        'y_train':    y_train,
        'y_test':     y_test,
        'n_train':    len(X_train),
        'n_test':     len(X_test),
        'n_features': X_train.shape[1],
    }


def encode_categoricals(df: pd.DataFrame,
                         cat_cols: list | None = None,
                         drop_first: bool = False) -> pd.DataFrame:
    """One-hot encode categorical columns with pd.get_dummies."""
    if cat_cols is None:
        cat_cols = df.select_dtypes(include='object').columns.tolist()
    if not cat_cols:
        return df.copy()
    encoded = pd.get_dummies(df, columns=cat_cols, drop_first=drop_first)
    # pandas 2.x returns bool dtype for dummy columns; convert to int
    bool_cols = encoded.select_dtypes(include='bool').columns.tolist()
    for c in bool_cols:
        encoded[c] = encoded[c].astype(int)
    return encoded


from sklearn.preprocessing import StandardScaler


def fit_scaler(X_train: pd.DataFrame) -> StandardScaler:
    """Fit a StandardScaler on training data only."""
    scaler = StandardScaler()
    scaler.fit(X_train)
    return scaler


def scale_features(scaler: StandardScaler,
                   X: pd.DataFrame) -> pd.DataFrame:
    """Transform X using a fitted scaler; return DataFrame with same columns."""
    scaled = scaler.transform(X)
    return pd.DataFrame(scaled, columns=X.columns, index=X.index)


from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score


def train_model(X_train: pd.DataFrame,
                y_train: pd.Series) -> LinearRegression:
    """Fit LinearRegression on training data."""
    model = LinearRegression()
    model.fit(X_train, y_train)
    return model


def evaluate_model(model: LinearRegression,
                   X_test: pd.DataFrame,
                   y_test: pd.Series) -> dict:
    """Return R², RMSE, n_test, and predictions array."""
    y_pred = model.predict(X_test)
    r2     = r2_score(y_test, y_pred)
    rmse   = float(np.sqrt(mean_squared_error(y_test, y_pred)))
    return {
        'r2':          round(float(r2), 4),
        'rmse':        round(rmse, 2),
        'n_test':      len(y_test),
        'predictions': y_pred,
    }


class FeatureEngineer:
    """
    End-to-end preprocessing pipeline: encode → scale → split.

    Usage:
        fe    = FeatureEngineer(target_col='price')
        split = fe.fit_transform(df)
        X_new = fe.transform(new_df)
    """

    def __init__(self, target_col: str,
                 cat_cols: list | None = None,
                 scale: bool = True):
        self.target_col   = target_col
        self.cat_cols     = cat_cols
        self.scale        = scale
        self._scaler      = None
        self._feature_cols = None

    def fit_transform(self, df: pd.DataFrame,
                      test_size: float = 0.2,
                      random_state: int = 42) -> dict:
        """Encode, scale (fit on train), split. Return split dict."""
        encoded             = encode_categoricals(df, cat_cols=self.cat_cols)
        X, y                = prepare_features(encoded, self.target_col,
                                               numeric_only=False)
        self._feature_cols  = X.columns.tolist()
        if self.scale:
            self._scaler = fit_scaler(X)
            X            = scale_features(self._scaler, X)
        return split_data(X, y, test_size=test_size,
                          random_state=random_state)

    def transform(self, df: pd.DataFrame) -> pd.DataFrame:
        """Apply fitted encoding + scaling to new data."""
        encoded = encode_categoricals(df, cat_cols=self.cat_cols)
        X       = encoded.reindex(columns=self._feature_cols, fill_value=0)
        if self.scale and self._scaler is not None:
            X = scale_features(self._scaler, X)
        return X

## Your Pipeline

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

df = make_regression_data(200)

# TODO: fe    = FeatureEngineer(target_col='price')
# TODO: split = fe.fit_transform(df)

# TODO: model  = train_model(split['X_train'], split['y_train'])
# TODO: result = evaluate_model(model, split['X_test'], split['y_test'])

# TODO: r2   = result['r2']
# TODO: rmse = result['rmse']
# TODO: print(f'R\u00b2 = {r2:.4f}')
# TODO: print(f'RMSE = ${rmse:,.2f}')

# TODO: # Feature coefficients
# TODO: coef_df = pd.DataFrame({
#     'feature': split['X_train'].columns.tolist(),
#     'coefficient': model.coef_,
# }).sort_values('coefficient', key=abs, ascending=False)
# TODO: print(coef_df.to_string(index=False))

# TODO: # Save scatter: actual vs predicted
# TODO: fig, ax = plt.subplots(figsize=(7, 5))
# TODO: ax.scatter(split['y_test'], result['predictions'], alpha=0.7)
# TODO: lo = min(split['y_test'].min(), result['predictions'].min())
# TODO: hi = max(split['y_test'].max(), result['predictions'].max())
# TODO: ax.plot([lo, hi], [lo, hi], 'r--', label='perfect fit')
# TODO: ax.set_xlabel('Actual Price ($)')
# TODO: ax.set_ylabel('Predicted Price ($)')
# TODO: ax.set_title(f'Actual vs Predicted  R\u00b2={r2:.3f}  RMSE=${rmse:,.0f}')
# TODO: ax.legend()
# TODO: fig.savefig('predictive_model.png', bbox_inches='tight', dpi=100)
# TODO: plt.close('all')
# TODO: print('Chart saved: predictive_model.png')

## Checks

In [ ]:
import os

def _run_project_checks():
    total = 5
    passed = 0

    # Check 1: r2 defined
    try:
        assert 'r2' in globals(), 'r2 not defined — run evaluate_model'
        assert isinstance(r2, (int, float)), f'r2 must be numeric, got {type(r2).__name__}'
        passed += 1; print(f'\u2705 Check 1: r2 = {r2:.4f}')
    except Exception as e:
        print(f'\u274c Check 1: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 2: r2 > 0.9 on housing data
    try:
        assert r2 > 0.9, f'R\u00b2 should be > 0.9 on housing data, got {r2:.4f}'
        passed += 1; print(f'\u2705 Check 2: R\u00b2={r2:.4f} > 0.90')
    except Exception as e:
        print(f'\u274c Check 2: {e}')

    # Check 3: rmse defined and positive
    try:
        assert 'rmse' in globals(), 'rmse not defined — run evaluate_model'
        assert isinstance(rmse, (int, float)) and rmse > 0, \
            f'rmse must be a positive number, got {rmse}'
        passed += 1; print(f'\u2705 Check 3: rmse = ${rmse:,.2f}')
    except Exception as e:
        print(f'\u274c Check 3: {e}')

    # Check 4: coef_df defined
    try:
        assert 'coef_df' in globals(), 'coef_df not defined — build coefficient table'
        assert isinstance(coef_df, pd.DataFrame)
        assert 'feature' in coef_df.columns and 'coefficient' in coef_df.columns
        passed += 1; print(f'\u2705 Check 4: coef_df with {len(coef_df)} features')
    except Exception as e:
        print(f'\u274c Check 4: {e}')

    # Check 5: chart saved
    try:
        assert os.path.exists('predictive_model.png'), \
            'predictive_model.png not found — save with fig.savefig()'
        assert os.path.getsize('predictive_model.png') > 1000, \
            'predictive_model.png looks empty'
        passed += 1; print('\u2705 Check 5: predictive_model.png saved')
    except Exception as e:
        print(f'\u274c Check 5: {e}')

    if passed == total:
        print('\U0001f389 Project complete!')
    print(f'\nScore: {passed}/{total}')


_run_project_checks()

## Bonus Challenges

- Try `drop_first=True` in the FeatureEngineer and see if R² changes
- Try `scale=False` — does scaling affect LinearRegression accuracy?
- Add a new synthetic feature: `area_per_bedroom = area / bedrooms` before calling fit_transform
- Use `ollama.chat` (Day 40 pattern) to narrate the model performance: pass R², RMSE, and the top 3 features by coefficient into a prompt
- Replace LinearRegression with `sklearn.tree.DecisionTreeRegressor(max_depth=5)` — same interface, does R² improve?